# 02 - Baselines

TF-IDF + Logistic Regression and TF-IDF + Linear SVM / Naive Bayes, plus the agreement experiments (all data vs high agreement only vs weighted loss).

Everything is tuned on val. Test is only used once at the end for the final models.

Runs fine on a CPU runtime.

In [ ]:
import os, sys

# on colab clone the repo, locally just move up from notebooks/
if "google.colab" in sys.modules:
    if not os.path.exists("/content/NLP_and_Language_technologies-Group-9"):
        !git clone https://github.com/Samkwizera/NLP_and_Language_technologies-Group-9.git /content/NLP_and_Language_technologies-Group-9
    %cd /content/NLP_and_Language_technologies-Group-9
    !pip install -q -r requirements.txt
    # csvs aren't in git, so copy them from drive (MyDrive/nlp_data/)
    from google.colab import drive
    drive.mount("/content/drive")
    import glob, shutil
    # zindi downloads come as "train (1).csv" etc, the code expects Train.csv / Test.csv
    for f in glob.glob("/content/drive/MyDrive/nlp_data/*.csv"):
        name = os.path.basename(f).lower()
        if name.startswith("train"):
            shutil.copy(f, "data/Train.csv")
        elif name.startswith("test"):
            shutil.copy(f, "data/Test.csv")
    if not os.path.exists("data/Train.csv"):
        raise FileNotFoundError("put the train/test csvs in MyDrive/nlp_data/")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
os.listdir("data")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.split import load_split
from src.utils import set_seed, ensure_dirs, add_takeaway
from src.evaluate import compute_metrics, plot_confusion_matrix, plot_learning_curve, export_errors
from src.models.baselines import run_experiment, learning_curve, confidence

set_seed()
ensure_dirs()
pd.set_option("display.max_colwidth", 120)

train, val, test = load_split()
for name, df in [("train", train), ("val", val), ("test", test)]:
    print(name, len(df), (df["label"].value_counts(normalize=True).sort_index() * 100).round(1).tolist())

## 1. The shared split

The split is in `splits/split_seed42.csv` (tweet ids only) and everyone loads it with `load_split()`. It's 70/15/15, seed 42, stratified on label and agreement together, and all copies of a duplicated tweet are in the same set, so nothing leaks from train into val/test. To rebuild it: `python -m src.split`.

## 2. Baseline experiments

Each run is logged to `results/experiments.csv`. We also keep per-class F1 here since the log only has the main metrics.

In [ ]:
runs = [
    dict(exp_id="B1", model="logreg", features="word",
         change="word 1-2 grams only", reason="simplest starting point"),
    dict(exp_id="B2", model="logreg", features="word+char",
         change="added char 2-5 grams", reason="half the vocab appears once, char n-grams should help with typos and rare words"),
    dict(exp_id="B3", model="logreg", features="word+char", class_weight="balanced",
         change="balanced class weights", reason="negative is only 10% of the data"),
    dict(exp_id="N1", model="nb", features="word+char", C=0.3,
         change="complement naive bayes", reason="nb is a common strong baseline for short text, complement nb handles imbalance better"),
    dict(exp_id="S1", model="svm", features="word+char", class_weight="balanced", C=0.1,
         change="linear svm instead of logreg", reason="linear svms usually do well on sparse tf-idf features"),
]

results = {}
for run in runs:
    pipe, preds, m = run_experiment(train_df=train, eval_df=val, **run)
    results[run["exp_id"]] = m
    print(run["exp_id"], m)

pd.DataFrame(results).T

### Tuning C

Small grid on val for the two linear models (both with word+char features and balanced weights).

In [ ]:
grid = {"logreg": [0.3, 1, 3, 10, 30], "svm": [0.03, 0.1, 0.3, 1]}
tuning = []
for model, Cs in grid.items():
    for C in Cs:
        _, _, m = run_experiment("tmp", train, val, model=model, C=C, class_weight="balanced", log=False)
        tuning.append({"model": model, "C": C, **m})
tuning = pd.DataFrame(tuning)
tuning

In [ ]:
best_C = tuning.loc[tuning.groupby("model")["macro_f1"].idxmax()].set_index("model")["C"].to_dict()
print(best_C)

for exp_id, model in [("B4", "logreg"), ("S2", "svm")]:
    _, _, m = run_experiment(exp_id, train, val, model=model, C=best_C[model], class_weight="balanced",
                             change=f"tuned C={best_C[model]}", reason="grid search on val macro-F1")
    results[exp_id] = m

pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)

**Notes:** _(fill in after running: does char n-grams help? does class weighting help negative F1? logreg vs svm vs nb)_

## 3. Agreement experiments

Same setup as the tuned models, only the training data changes. Val stays the same (all agreement levels) so the scores are comparable.

- `all`: every training tweet
- `high`: only agreement = 1
- `drop_333`: drop the 239-ish no-majority tweets (all labelled negative)
- `weighted`: keep everything, weight each tweet by its agreement in the loss

In [ ]:
modes = {
    "all": ("all training data", "reference for the agreement runs"),
    "high": ("only agreement = 1", "cleaner labels, but loses ~63% of negatives"),
    "drop_333": ("dropped agreement = 0.333", "those tweets had no majority and were all defaulted to negative"),
    "weighted": ("loss weighted by agreement", "keep all data but trust uncertain labels less"),
}

agree_results = []
i = 1
for model in ["logreg", "svm"]:
    for mode, (change, reason) in modes.items():
        exp_id = f"A{i}"
        _, _, m = run_experiment(exp_id, train, val, model=model, C=best_C[model], class_weight="balanced",
                                 agreement=mode, change=f"{model}: {change}", reason=reason)
        agree_results.append({"exp_id": exp_id, "model": model, "agreement": mode, **m})
        i += 1

agree_results = pd.DataFrame(agree_results)
agree_results

In [ ]:
agree_results.pivot(index="agreement", columns="model", values=["macro_f1", "f1_negative"]).loc[list(modes)]

In [ ]:
# also check how each setting does on val tweets split by their agreement,
# to see if training on clean labels only helps on the easy tweets
rows = []
for model in ["logreg", "svm"]:
    for mode in modes:
        pipe, preds, _ = run_experiment("tmp", train, val, model=model, C=best_C[model],
                                        class_weight="balanced", agreement=mode, log=False)
        for a, part in val.assign(pred=preds).groupby(val["agreement"].round(3)):
            rows.append({"model": model, "agreement_mode": mode, "val_agreement": a,
                         "macro_f1": compute_metrics(part["label"], part["pred"])["macro_f1"], "n": len(part)})
pd.DataFrame(rows).pivot_table(index=["model", "agreement_mode"], columns="val_agreement", values="macro_f1")

**Notes:** _(fill in after running: which agreement setting wins, what happens to negative F1 with `high`)_

## 4. Final baseline models

For each model we keep the agreement setting with the best val macro-F1.

In [ ]:
best = agree_results.loc[agree_results.groupby("model")["macro_f1"].idxmax()].set_index("model")
final = {model: dict(model=model, C=best_C[model], class_weight="balanced", agreement=best.loc[model, "agreement"])
         for model in ["logreg", "svm"]}
final

In [ ]:
fitted = {}
for model, cfg in final.items():
    pipe, preds, m = run_experiment("tmp", train, val, log=False, **cfg)
    fitted[model] = pipe
    print(model, m)
    plot_confusion_matrix(val["label"], preds, f"{model} (val)", name=f"baseline_{model}_val_cm")

In [ ]:
for model, cfg in final.items():
    sizes, tr, va = learning_curve(train, val, model=model, C=cfg["C"], class_weight="balanced")
    plot_learning_curve(sizes, tr, va, f"Learning curve: tf-idf + {model}", name=f"baseline_{model}_learning_curve")

**Notes:** _(fill in after running: is val still going up at full size? big gap between train and val?)_

## 5. Error analysis (val)

In [ ]:
errors = {}
for model, pipe in fitted.items():
    preds = pipe.predict(val["safe_text"])
    errors[model] = export_errors(val, preds, confidence(pipe, val["safe_text"]), f"baseline_{model}")
    print(model, len(errors[model]), "errors saved to results/errors/")

err = errors["logreg"]
pd.crosstab(err["true"], err["predicted"])

In [ ]:
# error rate by annotator agreement, hashtags and replies
v = val.assign(pred=fitted["logreg"].predict(val["safe_text"]))
v["wrong"] = v["pred"] != v["label"]
v["has_hashtag"] = v["safe_text"].str.contains("#")
v["has_user"] = v["safe_text"].str.contains("<user>", regex=False)
for col in ["agreement", "has_hashtag", "has_user"]:
    print(v.groupby(v[col].round(3) if col == "agreement" else v[col])["wrong"].agg(["mean", "size"]).round(3), "\n")

In [ ]:
# most confident mistakes
err[["text", "true", "predicted", "agreement", "confidence"]].head(20)

In [ ]:
# true positive tweets predicted negative: are these the angry pro-vaccine tweets from the eda?
err[(err["true"] == 1) & (err["predicted"] == -1)][["text", "agreement", "confidence"]].head(15)

**Notes:** _(fill in after running: main error types, examples)_

## 6. Test set

Only run once, after everything above is decided.

In [ ]:
test_results = {}
for (model, cfg), exp_id in zip(final.items(), ["BT1", "BT2"]):
    _, preds, m = run_experiment(exp_id, train, test, **cfg,
                                 change=f"final {model} on test (agreement={cfg['agreement']})",
                                 reason="final evaluation of the best val config")
    test_results[model] = m
    plot_confusion_matrix(test["label"], preds, f"{model} (test)", name=f"baseline_{model}_test_cm")
pd.DataFrame(test_results).T

In [ ]:
pd.read_csv(config.EXPERIMENTS_CSV)

In [ ]:
# takeaways go into results/experiments.csv, fill these in after reading the results
takeaways = {
    # "B1": "...",
}
for exp_id, text in takeaways.items():
    add_takeaway(exp_id, text)